# Portfolio Optimization — Institutional Analysis Notebook

**Methodology:** Markowitz Mean-Variance Optimization with multi-factor fundamental screening  
**Universe:** S&P 500  
**Outputs:** Max Sharpe · Min Volatility · Max Sortino portfolios

---

> This notebook walks through the full pipeline interactively. Each section can be run independently once the universe and prices are loaded. All charts are rendered inline; results are also exported to `output/`.

## 0 · Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("."))

import warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%H:%M:%S",
)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 120

from config import (
    DATE_START, DATE_END, RISK_FREE_RATE,
    N_PORTFOLIOS, MIN_FILTER_PASSES, QUANTILE_THRESHOLD,
)
from src.universe   import load_sp500_components, download_prices
from src.screening  import StockFilter, FundamentalScreener, fetch_all_fundamentals
from src.optimizer  import generate_portfolios, optimize_max_sharpe, optimize_min_volatility, optimize_max_sortino
from src.risk       import full_risk_report
from src.charts     import (
    plot_efficient_frontier, plot_weights,
    plot_correlation_heatmap, plot_nav_vs_benchmark, plot_drawdown,
)
from src.reporting  import export_portfolios, export_risk_metrics, export_screened_stocks

print("Environment ready")
print(f"  Universe period : {DATE_START.date()} -> {DATE_END.date()}")
print(f"  Risk-free rate  : {RISK_FREE_RATE:.2%}")
print(f"  Simulations     : {N_PORTFOLIOS:,}")

---
## 1 · Universe

We start from the full S&P 500 constituent list. The list is fetched from Wikipedia and cached locally for 24 hours so repeated runs avoid unnecessary HTTP requests.

In [ ]:
components = load_sp500_components()
tickers    = components["symbol"].tolist()
sector_map = dict(zip(components["symbol"], components["sector"]))

print(f"Universe: {len(tickers)} tickers across {components['sector'].nunique()} sectors\n")
components["sector"].value_counts().rename("# stocks").to_frame()

---
## 2 · Price History

Adjusted close prices are downloaded via **yfinance** and cached to `data/cache/`. Subsequent runs load from disk unless the cache is older than `CACHE_TTL_HOURS` (default: 24 h).

In [ ]:
prices_raw = download_prices(tickers, start=DATE_START, end=DATE_END)

print(f"Raw matrix : {prices_raw.shape[0]:,} trading days  x  {prices_raw.shape[1]:,} tickers")
print(f"Date range : {prices_raw.index[0].date()} -> {prices_raw.index[-1].date()}")
prices_raw.tail(3)

---
## 3 · Stage 1 Screening — Structural Filters

Two fast, price-based filters are applied before touching fundamental data:

| Filter | Logic |
|--------|-------|
| **Null** | Drop any ticker with at least one `NaN` in its price history |
| **Sharpe** | Drop any ticker whose annualized Sharpe ratio (rf = 0) is negative — persistent losers are excluded |

In [ ]:
sf = StockFilter()
prices          = sf.filtro_nulo(prices_raw)
prices, sharpes = sf.filtro_sharpe(prices)

print(f"After structural filters: {len(prices.columns)} stocks remain\n")
print("Top 10 by Sharpe ratio:")
sharpes[prices.columns].sort_values(ascending=False).head(10).round(3).to_frame("Sharpe")

---
## 4 · Stage 2 Screening — Fundamental Multi-Factor Filter

Each surviving ticker is evaluated against **12 financial metrics**. Fundamentals are fetched once per ticker and shared across all sub-filters.

Scoring is **sector-relative**: a stock earns one point per metric where it outperforms its sector peers. Only those surpassing `MIN_FILTER_PASSES` (default: 6 / 12) advance.

| Metric | Direction | Rationale |
|--------|-----------|-----------|
| Sharpe Ratio | ↑ | Risk-adjusted price momentum |
| Current Ratio | ↑ | Short-term liquidity |
| P/E Ratio | ±1σ band | Avoid extreme overvaluation and distressed names |
| EBITDA Margins | ↑ | Operating profitability |
| Debt / Equity | ↓ | Leverage risk |
| EV / EBITDA | ↓ | Valuation efficiency |
| Return on Assets | ↑ | Asset utilization |
| Return on Equity | ↑ | Equity profitability |
| Revenue Growth | ↑ | Top-line momentum |
| Earnings Growth | ↑ | Earnings quality |
| Beta | < mean + 1σ | Avoid excessive market sensitivity |
| Gross Margins | ↑ | Pricing power |

In [ ]:
fundamentals = fetch_all_fundamentals(prices.columns.tolist())

screener = FundamentalScreener()
prices, score_counts, fundamentals_df = screener.screen(prices, sharpes, fundamentals)
prices, stock_summary                 = screener.top_sector(prices, score_counts, fundamentals_df)

print(f"Stocks after fundamental screening: {len(prices.columns)}\n")
stock_summary.head(20)

In [ ]:
# Score distribution across sectors
stock_summary.groupby("sector")["score"].agg(["count", "mean", "min", "max"]).rename(
    columns={"count": "# stocks", "mean": "avg score", "min": "min score", "max": "max score"}
).round(1)

---
## 5 · Correlation Structure

Before optimizing, we inspect the correlation matrix of the screened universe. Low-to-moderate cross-correlations are precisely what Markowitz diversification exploits.

In [ ]:
fig = plot_correlation_heatmap(prices)
plt.show()

---
## 6 · Portfolio Optimization

We simulate **30,000 random weight allocations** to map the feasible risk-return space, then run constrained SLSQP optimization to find three distinct optimal portfolios:

| Portfolio | Objective |
|-----------|-----------|
| **Max Sharpe** | Maximize `(r − rf) / σ` |
| **Min Volatility** | Minimize annualized σ |
| **Max Sortino** | Maximize `(r − rf) / σ⁻` (downside deviation only) |

In [ ]:
import yfinance as yf

# Download benchmark once — reused by all risk reports
spy_raw    = yf.download("SPY", start=prices.index[0], end=prices.index[-1], auto_adjust=True, progress=False)["Close"]
spy_prices = spy_raw.squeeze()

# Simulate + optimize
sim_df, _      = generate_portfolios(prices, N_PORTFOLIOS, RISK_FREE_RATE)
p_max_sharpe   = optimize_max_sharpe(prices, RISK_FREE_RATE)
p_min_vol      = optimize_min_volatility(prices)
p_max_sortino  = optimize_max_sortino(prices, RISK_FREE_RATE)
portfolios     = [p_max_sharpe, p_min_vol, p_max_sortino]

print("Optimization complete.\n")
for p in portfolios:
    print(f"  {p['label']:<18}  return={p['annual_return']:.1%}  vol={p['volatility']:.1%}  sharpe={p['sharpe']:.2f}")

In [ ]:
fig = plot_efficient_frontier(sim_df, portfolios)
plt.show()

---
## 7 · Portfolio Weights

In [ ]:
for p in portfolios:
    fig = plot_weights(p, sector_map)
    plt.show()

---
## 8 · Risk Analytics

Full institutional risk decomposition for each portfolio. All metrics are computed on realized daily returns of the actual portfolio weights.

| Metric | Description |
|--------|-------------|
| **Annual Return** | CAGR over full backtest window |
| **Annual Volatility** | Annualized std dev of daily returns |
| **Sharpe Ratio** | `(CAGR − rf) / σ` |
| **Sortino Ratio** | `(CAGR − rf) / σ⁻` — penalizes only downside deviation |
| **Max Drawdown** | Largest peak-to-trough decline |
| **VaR 95%** | Worst daily loss not exceeded 95% of the time |
| **CVaR 95%** | Mean loss in the worst 5% of days (Expected Shortfall) |
| **Beta vs SPY** | Sensitivity to S&P 500 movements |
| **Calmar Ratio** | `CAGR / \|Max Drawdown\|` |

In [ ]:
risk_reports = [
    full_risk_report(p, prices, benchmark_prices=spy_prices, risk_free_rate=RISK_FREE_RATE)
    for p in portfolios
]

summary = pd.DataFrame([
    {
        "Portfolio":           r["label"],
        "Annual Return":       f"{r['annual_return']:.1%}",
        "Annual Volatility":   f"{r['annual_volatility']:.1%}",
        "Sharpe":              f"{r['sharpe_ratio']:.2f}",
        "Sortino":             f"{r['sortino_ratio']:.2f}",
        "Max Drawdown":        f"{r['max_drawdown']:.1%}",
        "VaR 95% (daily)":     f"{r['var_95']:.2%}",
        "CVaR 95% (daily)":    f"{r['cvar_95']:.2%}",
        "Beta vs SPY":         f"{r['beta']:.2f}",
        "Calmar":              f"{r['calmar_ratio']:.2f}",
    }
    for r in risk_reports
]).set_index("Portfolio")

summary

---
## 9 · Historical Performance vs Benchmark

In [ ]:
for r in risk_reports:
    fig = plot_nav_vs_benchmark(r["_daily_returns"], r["_benchmark_returns"], label=r["label"])
    plt.show()

---
## 10 · Drawdown Analysis

In [ ]:
for r in risk_reports:
    fig = plot_drawdown(r["_daily_returns"], label=r["label"])
    plt.show()

---
## 11 · Export Results

All outputs are written to `output/`:

| File | Contents |
|------|----------|
| `portfolios.csv` | Weight allocation of all 3 portfolios |
| `risk_metrics.csv` | Comparative risk metrics table |
| `screened_stocks.csv` | Stocks that passed screening with sector and score |
| `*.png` | All charts generated above |

In [ ]:
path_portfolios = export_portfolios(portfolios)
path_risk       = export_risk_metrics(risk_reports)
path_stocks     = export_screened_stocks(stock_summary)

print("Exports complete:")
print(f"  {path_portfolios}")
print(f"  {path_risk}")
print(f"  {path_stocks}")

---

## Appendix — Configuration Reference

All tunable parameters live in `config.py`. No code changes needed for common adjustments.

| Parameter | Default | Description |
|-----------|---------|-------------|
| `RISK_FREE_RATE` | `0.0425` | Annual risk-free rate used in Sharpe / Sortino |
| `TRADING_DAYS` | `252` | Annualization factor |
| `N_PORTFOLIOS` | `30,000` | Monte Carlo simulations |
| `MIN_FILTER_PASSES` | `6` | Minimum fundamental filters a stock must pass |
| `QUANTILE_THRESHOLD` | `0.80` | Sector-relative percentile cutoff |
| `MAX_WEIGHT_PER_STOCK` | `1.0` | Per-asset weight cap (set `0.20` to limit to 20%) |
| `CACHE_TTL_HOURS` | `24` | Hours before cached data expires |
| `DATE_START` | `2015-01-01` | Start of price history |